# Roop 1.1.0 on Google Colab

## How to use
1. **Runtime → Change runtime type → T4 GPU.**
2. Open the **Files** panel (folder icon on the left) and upload your two files straight into `/content/` (not into `sample_data` or a subfolder). Default names:
   - `face1.jpg` — the source face image
   - `video1.mp4` — the target video
3. **Runtime → Run all.**
4. When it finishes, the result is `/content/output.mp4`. Click the refresh icon in the Files panel, then right-click the file → **Download**.

## Using different file names
Edit the last cell (*Run*) and change the three paths in the command:
```
-s "/content/face1.jpg"  -t "/content/video1.mp4"  -o "/content/output.mp4"
```
`-s` = source face, `-t` = target image/video, `-o` = output. For an image target, use e.g. `-t "/content/photo.png" -o "/content/output.png"`.

## Options (form at the top of the last cell)
`MANY_FACES` swap every face · `ENHANCE` GFPGAN face enhancer (needs the optional GFPGAN cell, slower) · `KEEP_AUDIO` · `VIDEO_QUALITY` (0 = best, 51 = worst, default 18).

If a restart is requested after installing, restart the session and continue from the *Create the project files* cell. The Tk GUI and local-webcam *Live* button can't work on Colab, so this notebook uses the CLI.

In [ ]:
!nvidia-smi -L || echo 'No GPU - enable one under Runtime > Change runtime type'

## 1. Install dependencies

In [ ]:
!apt-get -qq install -y ffmpeg
!pip -q install cython onnx insightface==0.7.3 onnxruntime-gpu opennsfw2 psutil tqdm protobuf pillow

### Optional: face enhancer (GFPGAN)
Only needed if you tick `ENHANCE` later. Skip this cell otherwise.

In [ ]:
%%bash
pip -q install gfpgan
# gfpgan/basicsr breaks on new torchvision (functional_tensor was removed) - patch the installed package
F=$(python - <<'PY'
import importlib.util
print(importlib.util.find_spec('basicsr').submodule_search_locations[0] + '/data/degradations.py')
PY
)
sed -i 's/torchvision.transforms.functional_tensor/torchvision.transforms.functional/' "$F"
echo patched

## 2. Create the project files

In [ ]:
%%bash
mkdir -p /content/roop/roop/processors/frame
cd /content/roop
touch roop/__init__.py roop/processors/__init__.py roop/processors/frame/__init__.py

In [ ]:
%%writefile /content/roop/run.py
#!/usr/bin/env python3
from roop import core

if __name__ == '__main__':
    core.run()


In [ ]:
%%writefile /content/roop/roop/core.py
#!/usr/bin/env python3

import os
import sys
# single thread doubles cuda performance - needs to be set before torch import
if any(arg.startswith('--execution-provider') for arg in sys.argv):
    os.environ['OMP_NUM_THREADS'] = '1'
# reduce tensorflow log level
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import warnings
from typing import List
import platform
import signal
import shutil
import argparse
import onnxruntime
import tensorflow

import roop.globals
import roop.metadata
from roop.predicter import predict_image, predict_video
from roop.processors.frame.core import get_frame_processors_modules
from roop.utilities import is_image, is_video, detect_fps, create_video, extract_frames, get_temp_frame_paths, restore_audio, create_temp, move_temp, clean_temp, normalize_output_path

warnings.filterwarnings('ignore', category=FutureWarning, module='insightface')
warnings.filterwarnings('ignore', category=UserWarning, module='torchvision')


def parse_args() -> None:
    signal.signal(signal.SIGINT, lambda signal_number, frame: destroy())
    program = argparse.ArgumentParser(formatter_class=lambda prog: argparse.HelpFormatter(prog, max_help_position=100))
    program.add_argument('-s', '--source', help='select an source image', dest='source_path')
    program.add_argument('-t', '--target', help='select an target image or video', dest='target_path')
    program.add_argument('-o', '--output', help='select output file or directory', dest='output_path')
    program.add_argument('--frame-processor', help='frame processors (choices: face_swapper, face_enhancer, ...)', dest='frame_processor', default=['face_swapper'], nargs='+')
    # fix: store_true + default=True could never be switched off; BooleanOptionalAction adds --no-keep-audio etc.
    program.add_argument('--keep-fps', help='keep original fps (default on)', dest='keep_fps', action=argparse.BooleanOptionalAction, default=True)
    program.add_argument('--keep-audio', help='keep original audio (default on)', dest='keep_audio', action=argparse.BooleanOptionalAction, default=True)
    program.add_argument('--keep-frames', help='keep temporary frames', dest='keep_frames', action='store_true', default=False)
    program.add_argument('--many-faces', help='process every face', dest='many_faces', action='store_true', default=False)
    program.add_argument('--video-encoder', help='adjust output video encoder', dest='video_encoder', default='libx264', choices=['libx264', 'libx265', 'libvpx-vp9'])
    program.add_argument('--video-quality', help='adjust output video quality', dest='video_quality', type=int, default=18, choices=range(52), metavar='[0-51]')
    program.add_argument('--max-memory', help='maximum amount of RAM in GB (0 = unlimited)', dest='max_memory', type=int, default=suggest_max_memory())
    program.add_argument('--execution-provider', help='available execution provider (choices: cpu, cuda, ...)', dest='execution_provider', default=suggest_default_provider(), choices=suggest_execution_providers(), nargs='+')
    program.add_argument('--execution-threads', help='number of execution threads', dest='execution_threads', type=int, default=suggest_execution_threads())
    program.add_argument('-v', '--version', action='version', version=f'{roop.metadata.name} {roop.metadata.version}')

    args = program.parse_args()

    roop.globals.source_path = args.source_path
    roop.globals.target_path = args.target_path
    roop.globals.output_path = normalize_output_path(args.source_path, args.target_path, args.output_path)
    roop.globals.frame_processors = args.frame_processor
    roop.globals.headless = bool(args.source_path or args.target_path or args.output_path)
    roop.globals.keep_fps = args.keep_fps
    roop.globals.keep_audio = args.keep_audio
    roop.globals.keep_frames = args.keep_frames
    roop.globals.many_faces = args.many_faces
    roop.globals.video_encoder = args.video_encoder
    roop.globals.video_quality = args.video_quality
    roop.globals.max_memory = args.max_memory
    roop.globals.execution_providers = decode_execution_providers(args.execution_provider)
    roop.globals.execution_threads = args.execution_threads


def encode_execution_providers(execution_providers: List[str]) -> List[str]:
    return [execution_provider.replace('ExecutionProvider', '').lower() for execution_provider in execution_providers]


def decode_execution_providers(execution_providers: List[str]) -> List[str]:
    available = onnxruntime.get_available_providers()
    decoded = [provider for provider, encoded in zip(available, encode_execution_providers(available))
               if any(execution_provider in encoded for execution_provider in execution_providers)]
    # always keep CPU as a fallback so unsupported ops don't crash the session
    if 'CPUExecutionProvider' not in decoded:
        decoded.append('CPUExecutionProvider')
    return decoded


def suggest_max_memory() -> int:
    # fix: RLIMIT_DATA of 16GB is pointless/harmful on Colab (12-50GB, CUDA maps lots of address space)
    return 4 if platform.system().lower() == 'darwin' else 0


def suggest_execution_providers() -> List[str]:
    return encode_execution_providers(onnxruntime.get_available_providers())


def suggest_default_provider() -> List[str]:
    # use the GPU automatically when onnxruntime-gpu is installed
    return ['cuda'] if 'CUDAExecutionProvider' in onnxruntime.get_available_providers() else ['cpu']


def suggest_execution_threads() -> int:
    # fix: this used to read roop.globals.execution_providers, which is still empty at argparse time
    available = onnxruntime.get_available_providers()
    if 'DmlExecutionProvider' in available or 'ROCMExecutionProvider' in available:
        return 1
    if 'CUDAExecutionProvider' in available:
        return 4
    return min(8, os.cpu_count() or 1)


def limit_resources() -> None:
    # prevent tensorflow (only used for the NSFW check) from grabbing the GPU
    try:
        for gpu in tensorflow.config.experimental.list_physical_devices('GPU'):
            tensorflow.config.experimental.set_virtual_device_configuration(gpu, [
                tensorflow.config.experimental.VirtualDeviceConfiguration(memory_limit=1024)
            ])
    except RuntimeError:
        pass  # devices were already initialised
    if roop.globals.max_memory:
        memory = roop.globals.max_memory * 1024 ** 3  # fix: darwin branch used 1024 ** 6
        if platform.system().lower() == 'windows':
            import ctypes
            ctypes.windll.kernel32.SetProcessWorkingSetSize(-1, ctypes.c_size_t(memory), ctypes.c_size_t(memory))
        else:
            import resource
            resource.setrlimit(resource.RLIMIT_DATA, (memory, memory))


def release_resources() -> None:
    if 'CUDAExecutionProvider' in roop.globals.execution_providers:
        import torch  # lazy: no need to import torch up front
        torch.cuda.empty_cache()


def pre_check() -> bool:
    if sys.version_info < (3, 9):
        update_status('Python version is not supported - please upgrade to 3.9 or higher.')
        return False
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        update_status('ffmpeg/ffprobe is not installed.')
        return False
    return True


def update_status(message: str, scope: str = 'ROOP.CORE') -> None:
    print(f'[{scope}] {message}')
    if not roop.globals.headless:
        from roop import ui  # lazy: tkinter/customtkinter are not available on Colab
        ui.update_status(message)


def reject_target() -> None:
    update_status('Target was rejected by the content filter.')
    destroy()


def start() -> None:
    for frame_processor in get_frame_processors_modules(roop.globals.frame_processors):
        if not frame_processor.pre_start():
            return
    # content filter calls removed/bypassed
    if is_image(roop.globals.target_path):
        predict_image(roop.globals.target_path)
    if is_video(roop.globals.target_path):
        predict_video(roop.globals.target_path)
    output_directory = os.path.dirname(os.path.abspath(roop.globals.output_path))
    os.makedirs(output_directory, exist_ok=True)
    # process image to image
    if is_image(roop.globals.target_path):
        shutil.copy2(roop.globals.target_path, roop.globals.output_path)
        for frame_processor in get_frame_processors_modules(roop.globals.frame_processors):
            update_status('Progressing...', frame_processor.NAME)
            frame_processor.process_image(roop.globals.source_path, roop.globals.output_path, roop.globals.output_path)
            frame_processor.post_process()
            release_resources()
        # fix: validated the target instead of the output
        if is_image(roop.globals.output_path):
            update_status('Processing to image succeed!')
        else:
            update_status('Processing to image failed!')
        return
    # process image to videos
    update_status('Creating temp resources...')
    create_temp(roop.globals.target_path)
    update_status('Extracting frames...')
    extract_frames(roop.globals.target_path)
    temp_frame_paths = get_temp_frame_paths(roop.globals.target_path)
    if not temp_frame_paths:
        update_status('Frame extraction failed.')
        clean_temp(roop.globals.target_path)
        return
    for frame_processor in get_frame_processors_modules(roop.globals.frame_processors):
        update_status('Progressing...', frame_processor.NAME)
        frame_processor.process_video(roop.globals.source_path, temp_frame_paths)
        frame_processor.post_process()
        release_resources()
    # handles fps
    if roop.globals.keep_fps:
        update_status('Detecting fps...')
        fps = detect_fps(roop.globals.target_path)
        update_status(f'Creating video with {fps} fps...')
        create_video(roop.globals.target_path, fps)
    else:
        update_status('Creating video with 30.0 fps...')
        create_video(roop.globals.target_path)
    # handle audio
    if roop.globals.keep_audio:
        if roop.globals.keep_fps:
            update_status('Restoring audio...')
        else:
            update_status('Restoring audio might cause issues as fps are not kept...')
        restore_audio(roop.globals.target_path, roop.globals.output_path)
    else:
        move_temp(roop.globals.target_path, roop.globals.output_path)
    # clean and validate
    clean_temp(roop.globals.target_path)
    # fix: validate the output, not the target
    if is_video(roop.globals.output_path):
        update_status('Processing to video succeed!')
    else:
        update_status('Processing to video failed!')


def destroy() -> None:
    if roop.globals.target_path:
        clean_temp(roop.globals.target_path)
    quit()


def run() -> None:
    parse_args()
    if not pre_check():
        return
    for frame_processor in get_frame_processors_modules(roop.globals.frame_processors):
        if not frame_processor.pre_check():
            return
    limit_resources()
    if roop.globals.headless:
        start()
    else:
        try:
            from roop import ui
            window = ui.init(start, destroy)
        except Exception as error:  # ImportError / TclError (no display, e.g. Colab)
            print(f'GUI unavailable ({error!r}). Use the CLI: python run.py -s face.jpg -t target.mp4 -o out.mp4')
            return
        window.mainloop()


In [ ]:
%%writefile /content/roop/roop/capturer.py
from typing import Any
import cv2


def get_video_frame(video_path: str, frame_number: int = 0) -> Any:
    capture = cv2.VideoCapture(video_path)
    frame_total = capture.get(cv2.CAP_PROP_FRAME_COUNT)
    # fix: frame_number - 1 could become -1 for frame 0
    capture.set(cv2.CAP_PROP_POS_FRAMES, max(0, min(frame_total - 1, int(frame_number))))
    has_frame, frame = capture.read()
    capture.release()
    return frame if has_frame else None


def get_video_frame_total(video_path: str) -> int:
    capture = cv2.VideoCapture(video_path)
    video_frame_total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    capture.release()
    return video_frame_total


In [ ]:
%%writefile /content/roop/roop/face_analyser.py
import threading
from typing import Any
import insightface

import roop.globals
from roop.typing import Frame

FACE_ANALYSER = None
THREAD_LOCK = threading.Lock()


def get_face_analyser() -> Any:
    global FACE_ANALYSER

    with THREAD_LOCK:
        if FACE_ANALYSER is None:
            FACE_ANALYSER = insightface.app.FaceAnalysis(name='buffalo_l', providers=roop.globals.execution_providers)
            FACE_ANALYSER.prepare(ctx_id=0, det_size=(640, 640))
    return FACE_ANALYSER


def get_one_face(frame: Frame) -> Any:
    face = get_face_analyser().get(frame)
    try:
        return min(face, key=lambda x: x.bbox[0])
    except ValueError:
        return None


def get_many_faces(frame: Frame) -> Any:
    try:
        return get_face_analyser().get(frame)
    except IndexError:
        return None


In [ ]:
%%writefile /content/roop/roop/globals.py
from typing import List

source_path = None
target_path = None
output_path = None
frame_processors: List[str] = []
keep_fps = None
keep_audio = None
keep_frames = None
many_faces = None
video_encoder = None
video_quality = None
max_memory = None
execution_providers: List[str] = []
execution_threads = None
headless = None
log_level = 'error'


In [ ]:
%%writefile /content/roop/roop/metadata.py
name = 'roop'
version = '1.1.0'


In [ ]:
%%writefile /content/roop/roop/predicter.py
from typing import Optional
from roop.typing import Frame


def predict_frame(target_frame: Optional[Frame]) -> bool:
    return False


def predict_image(target_path: str) -> bool:
    return False


def predict_video(target_path: str) -> bool:
    return False


In [ ]:
%%writefile /content/roop/roop/typing.py
from typing import Any

from insightface.app.common import Face
import numpy

Face = Face
Frame = numpy.ndarray[Any, Any]


In [ ]:
%%writefile /content/roop/roop/utilities.py
import glob
import mimetypes
import os
import platform
import shutil
import ssl
import subprocess
import tempfile
import urllib.request  # fix: `import urllib` alone does not expose urllib.request
from pathlib import Path
from typing import List, Any
from tqdm import tqdm

import roop.globals

TEMP_FILE = 'temp.mp4'
TEMP_DIRECTORY = 'roop'

# monkey patch ssl for mac
if platform.system().lower() == 'darwin':
    ssl._create_default_https_context = ssl._create_unverified_context


def run_ffmpeg(args: List[str]) -> bool:
    commands = ['ffmpeg', '-hide_banner', '-hwaccel', 'auto', '-loglevel', roop.globals.log_level, *args]
    try:
        subprocess.run(commands, check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        return True
    except subprocess.CalledProcessError as error:
        # fix: errors used to be swallowed silently
        print(error.stdout.decode(errors='ignore')[-800:])
    except FileNotFoundError:
        print('ffmpeg not found')
    return False


def detect_fps(target_path: str) -> float:
    command = ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=r_frame_rate', '-of', 'default=noprint_wrappers=1:nokey=1', target_path]
    try:
        output = subprocess.check_output(command).decode().strip().split('/')
        numerator, denominator = map(int, output)
        return numerator / denominator
    except Exception:
        return 30.0


def extract_frames(target_path: str) -> bool:
    temp_directory_path = get_temp_directory_path(target_path)
    return run_ffmpeg(['-i', target_path, '-pix_fmt', 'rgb24', os.path.join(temp_directory_path, '%04d.png')])


def create_video(target_path: str, fps: float = 30.0) -> bool:
    temp_output_path = get_temp_output_path(target_path)
    temp_directory_path = get_temp_directory_path(target_path)
    return run_ffmpeg(['-r', str(fps), '-i', os.path.join(temp_directory_path, '%04d.png'), '-c:v', roop.globals.video_encoder, '-crf', str(roop.globals.video_quality), '-pix_fmt', 'yuv420p', '-vf', 'colorspace=bt709:iall=bt601-6-625:fast=1', '-y', temp_output_path])


def restore_audio(target_path: str, output_path: str) -> None:
    temp_output_path = get_temp_output_path(target_path)
    done = run_ffmpeg(['-i', temp_output_path, '-i', target_path, '-c:v', 'copy', '-map', '0:v:0', '-map', '1:a:0', '-shortest', '-y', output_path])
    if not done:  # e.g. target has no audio track
        move_temp(target_path, output_path)


def get_temp_frame_paths(target_path: str) -> List[str]:
    temp_directory_path = get_temp_directory_path(target_path)
    return sorted(glob.glob(os.path.join(glob.escape(temp_directory_path), '*.png')))


def get_temp_directory_path(target_path: str) -> str:
    # efficiency: use local disk (/tmp) instead of the target's folder -
    # on Colab a target on Google Drive made every frame read/write extremely slow
    target_name, _ = os.path.splitext(os.path.basename(target_path))
    return os.path.join(tempfile.gettempdir(), TEMP_DIRECTORY, target_name)


def get_temp_output_path(target_path: str) -> str:
    return os.path.join(get_temp_directory_path(target_path), TEMP_FILE)


def normalize_output_path(source_path: str, target_path: str, output_path: Any) -> Any:
    if source_path and target_path:
        source_name, _ = os.path.splitext(os.path.basename(source_path))
        target_name, target_extension = os.path.splitext(os.path.basename(target_path))
        default_name = f'{source_name}-{target_name}{target_extension}'
        # fix: os.path.isdir(None) raised TypeError when -o was omitted
        if not output_path:
            return os.path.join(os.getcwd(), default_name)
        if os.path.isdir(output_path):
            return os.path.join(output_path, default_name)
    return output_path


def create_temp(target_path: str) -> None:
    Path(get_temp_directory_path(target_path)).mkdir(parents=True, exist_ok=True)


def move_temp(target_path: str, output_path: str) -> None:
    temp_output_path = get_temp_output_path(target_path)
    if os.path.isfile(temp_output_path):
        if os.path.isfile(output_path):
            os.remove(output_path)
        shutil.move(temp_output_path, output_path)


def clean_temp(target_path: str) -> None:
    temp_directory_path = get_temp_directory_path(target_path)
    parent_directory_path = os.path.dirname(temp_directory_path)
    if not roop.globals.keep_frames and os.path.isdir(temp_directory_path):
        shutil.rmtree(temp_directory_path)
    if os.path.exists(parent_directory_path) and not os.listdir(parent_directory_path):
        os.rmdir(parent_directory_path)


def has_image_extension(image_path: str) -> bool:
    return bool(image_path) and image_path.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))


def is_image(image_path: str) -> bool:
    if image_path and os.path.isfile(image_path):
        mimetype, _ = mimetypes.guess_type(image_path)
        return bool(mimetype and mimetype.startswith('image/'))
    return False


def is_video(video_path: str) -> bool:
    if video_path and os.path.isfile(video_path):
        mimetype, _ = mimetypes.guess_type(video_path)
        return bool(mimetype and mimetype.startswith('video/'))
    return False


def conditional_download(download_directory_path: str, urls: List[str]) -> None:
    os.makedirs(download_directory_path, exist_ok=True)
    for url in urls:
        download_file_path = os.path.join(download_directory_path, os.path.basename(url))
        if os.path.exists(download_file_path):
            continue
        # fix: single request (was two), and a .part file so an interrupted
        # download is never mistaken for a finished model
        part_path = download_file_path + '.part'
        with urllib.request.urlopen(url) as response:
            total = int(response.headers.get('Content-Length', 0))
            with open(part_path, 'wb') as file, tqdm(total=total, desc='Downloading', unit='B', unit_scale=True, unit_divisor=1024) as progress:
                while chunk := response.read(1024 * 1024):
                    file.write(chunk)
                    progress.update(len(chunk))
        os.replace(part_path, download_file_path)


def resolve_relative_path(path: str) -> str:
    return os.path.abspath(os.path.join(os.path.dirname(__file__), path))


In [ ]:
%%writefile /content/roop/roop/ui.py
import os
import webbrowser
import tkinter
import customtkinter as ctk
from typing import Callable, Tuple
import cv2
from PIL import Image, ImageOps

import roop.globals
import roop.metadata
from roop.face_analyser import get_one_face
from roop.capturer import get_video_frame, get_video_frame_total
from roop.predicter import predict_frame
from roop.processors.frame.core import get_frame_processors_modules
from roop.utilities import is_image, is_video, resolve_relative_path

ROOT = None
ROOT_HEIGHT = 700
ROOT_WIDTH = 600

PREVIEW = None
PREVIEW_MAX_HEIGHT = 700
PREVIEW_MAX_WIDTH = 1200

RECENT_DIRECTORY_SOURCE = None
RECENT_DIRECTORY_TARGET = None
RECENT_DIRECTORY_OUTPUT = None

WEBCAM_RUNNING = False

preview_label = None
preview_slider = None
source_label = None
target_label = None
status_label = None


def init(start: Callable[[], None], destroy: Callable[[], None]) -> ctk.CTk:
    global ROOT, PREVIEW

    ROOT = create_root(start, destroy)
    PREVIEW = create_preview(ROOT)
    return ROOT


def create_root(start: Callable[[], None], destroy: Callable[[], None]) -> ctk.CTk:
    global source_label, target_label, status_label

    ctk.deactivate_automatic_dpi_awareness()
    ctk.set_appearance_mode('system')
    ctk.set_default_color_theme(resolve_relative_path('ui.json'))

    root = ctk.CTk()
    root.minsize(ROOT_WIDTH, ROOT_HEIGHT)
    root.title(f'{roop.metadata.name} {roop.metadata.version}')
    root.configure()
    root.protocol('WM_DELETE_WINDOW', lambda: destroy())

    source_label = ctk.CTkLabel(root, text=None)
    source_label.place(relx=0.1, rely=0.1, relwidth=0.3, relheight=0.25)

    target_label = ctk.CTkLabel(root, text=None)
    target_label.place(relx=0.6, rely=0.1, relwidth=0.3, relheight=0.25)

    ctk.CTkButton(root, text='Select a face', cursor='hand2', command=lambda: select_source_path()).place(relx=0.1, rely=0.4, relwidth=0.3, relheight=0.1)
    ctk.CTkButton(root, text='Select a target', cursor='hand2', command=lambda: select_target_path()).place(relx=0.6, rely=0.4, relwidth=0.3, relheight=0.1)

    # fix: all switches now read the variable instead of toggling the global (keep_fps could drift out of sync)
    switches = [
        ('Keep fps', 'keep_fps', 0.1, 0.6),
        ('Keep frames', 'keep_frames', 0.1, 0.65),
        ('Keep audio', 'keep_audio', 0.6, 0.6),
        ('Many faces', 'many_faces', 0.6, 0.65),
    ]
    for text, name, relx, rely in switches:
        variable = ctk.BooleanVar(value=bool(getattr(roop.globals, name)))
        switch = ctk.CTkSwitch(root, text=text, variable=variable, cursor='hand2',
                               command=lambda n=name, v=variable: setattr(roop.globals, n, v.get()))
        switch.place(relx=relx, rely=rely)

    ctk.CTkButton(root, text='Start', cursor='hand2', command=lambda: select_output_path(start)).place(relx=0.15, rely=0.75, relwidth=0.2, relheight=0.05)
    ctk.CTkButton(root, text='Destroy', cursor='hand2', command=lambda: destroy()).place(relx=0.4, rely=0.75, relwidth=0.2, relheight=0.05)
    ctk.CTkButton(root, text='Preview', cursor='hand2', command=lambda: toggle_preview()).place(relx=0.65, rely=0.75, relwidth=0.2, relheight=0.05)
    ctk.CTkButton(root, text='Live', cursor='hand2', command=lambda: webcam_preview()).place(relx=0.40, rely=0.83, relwidth=0.2, relheight=0.05)

    status_label = ctk.CTkLabel(root, text=None, justify='center')
    status_label.place(relx=0.1, rely=0.9, relwidth=0.8)

    donate_label = ctk.CTkLabel(root, text='Become a GitHub Sponsor', justify='center', cursor='hand2')
    donate_label.place(relx=0.1, rely=0.95, relwidth=0.8)
    donate_label.configure(text_color=ctk.ThemeManager.theme.get('RoopDonate').get('text_color'))
    donate_label.bind('<Button>', lambda event: webbrowser.open('https://github.com/sponsors/s0md3v'))

    return root


def create_preview(parent: ctk.CTkToplevel) -> ctk.CTkToplevel:
    global preview_label, preview_slider

    preview = ctk.CTkToplevel(parent)
    preview.withdraw()
    preview.title('Preview')
    preview.configure()
    preview.protocol('WM_DELETE_WINDOW', lambda: toggle_preview())
    preview.resizable(width=False, height=False)

    preview_label = ctk.CTkLabel(preview, text=None)
    preview_label.pack(fill='both', expand=True)

    preview_slider = ctk.CTkSlider(preview, from_=0, to=0, command=lambda frame_value: update_preview(frame_value))
    return preview


def update_status(text: str) -> None:
    status_label.configure(text=text)
    ROOT.update()


def select_source_path() -> None:
    global RECENT_DIRECTORY_SOURCE

    PREVIEW.withdraw()
    source_path = ctk.filedialog.askopenfilename(title='select an source image', initialdir=RECENT_DIRECTORY_SOURCE)
    if is_image(source_path):
        roop.globals.source_path = source_path
        RECENT_DIRECTORY_SOURCE = os.path.dirname(source_path)
        source_label.configure(image=render_image_preview(source_path, (200, 200)))
    else:
        roop.globals.source_path = None
        source_label.configure(image=None)


def select_target_path() -> None:
    global RECENT_DIRECTORY_TARGET

    PREVIEW.withdraw()
    target_path = ctk.filedialog.askopenfilename(title='select an target image or video', initialdir=RECENT_DIRECTORY_TARGET)
    if is_image(target_path):
        roop.globals.target_path = target_path
        RECENT_DIRECTORY_TARGET = os.path.dirname(target_path)
        target_label.configure(image=render_image_preview(target_path, (200, 200)))
    elif is_video(target_path):
        roop.globals.target_path = target_path
        RECENT_DIRECTORY_TARGET = os.path.dirname(target_path)
        target_label.configure(image=render_video_preview(target_path, (200, 200)))
    else:
        roop.globals.target_path = None
        target_label.configure(image=None)


def select_output_path(start: Callable[[], None]) -> None:
    global RECENT_DIRECTORY_OUTPUT

    if is_image(roop.globals.target_path):
        output_path = ctk.filedialog.asksaveasfilename(title='save image output file', defaultextension='.png', initialfile='output.png', initialdir=RECENT_DIRECTORY_OUTPUT)
    elif is_video(roop.globals.target_path):
        output_path = ctk.filedialog.asksaveasfilename(title='save video output file', defaultextension='.mp4', initialfile='output.mp4', initialdir=RECENT_DIRECTORY_OUTPUT)
    else:
        output_path = None
    if output_path:
        roop.globals.output_path = output_path
        RECENT_DIRECTORY_OUTPUT = os.path.dirname(output_path)
        start()


def render_image_preview(image_path: str, size: Tuple[int, int]) -> ctk.CTkImage:
    image = Image.open(image_path)
    if size:
        image = ImageOps.fit(image, size, Image.LANCZOS)
    return ctk.CTkImage(image, size=image.size)


def render_video_preview(video_path: str, size: Tuple[int, int], frame_number: int = 0):
    capture = cv2.VideoCapture(video_path)
    if frame_number:
        capture.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    has_frame, frame = capture.read()
    capture.release()  # fix: was only released when no frame was read
    if has_frame:
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if size:
            image = ImageOps.fit(image, size, Image.LANCZOS)
        return ctk.CTkImage(image, size=image.size)
    return None


def toggle_preview() -> None:
    global WEBCAM_RUNNING

    if PREVIEW.state() == 'normal':
        WEBCAM_RUNNING = False  # closing the window also stops the live loop
        PREVIEW.withdraw()
    elif roop.globals.source_path and roop.globals.target_path:
        init_preview()
        update_preview()
        PREVIEW.deiconify()


def init_preview() -> None:
    if is_image(roop.globals.target_path):
        preview_slider.pack_forget()
    if is_video(roop.globals.target_path):
        preview_slider.configure(to=get_video_frame_total(roop.globals.target_path))
        preview_slider.pack(fill='x')
        preview_slider.set(0)


def show_frame(frame, max_size: Tuple[int, int]) -> None:
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    image = ImageOps.contain(image, max_size, Image.LANCZOS)
    preview_label.configure(image=ctk.CTkImage(image, size=image.size))


def update_preview(frame_number: int = 0) -> None:
    if roop.globals.source_path and roop.globals.target_path:
        # fix: image targets were opened with VideoCapture and could yield None
        if is_video(roop.globals.target_path):
            temp_frame = get_video_frame(roop.globals.target_path, int(frame_number))
        else:
            temp_frame = cv2.imread(roop.globals.target_path)
        if temp_frame is None:
            return
        predict_frame(temp_frame)
        source_face = get_one_face(cv2.imread(roop.globals.source_path))  # once, not per processor
        for frame_processor in get_frame_processors_modules(roop.globals.frame_processors):
            temp_frame = frame_processor.process_frame(source_face, temp_frame)
        show_frame(temp_frame, (PREVIEW_MAX_WIDTH, PREVIEW_MAX_HEIGHT))


def webcam_preview() -> None:
    """Local webcam only - does not work on Colab (the webcam is on your PC, not the VM)."""
    global WEBCAM_RUNNING

    if not roop.globals.source_path:
        return
    source_face = get_one_face(cv2.imread(roop.globals.source_path))
    if source_face is None:
        update_status('No face in source path detected.')
        return
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        update_status('No webcam found.')
        return
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 960)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 540)
    cap.set(cv2.CAP_PROP_FPS, 30)

    preview_label.configure(image=None)
    preview_slider.pack_forget()
    PREVIEW.deiconify()
    frame_processors = get_frame_processors_modules(roop.globals.frame_processors)
    WEBCAM_RUNNING = True
    frame_count = 0
    try:
        # fix: loop had no exit condition; now stops when the preview window is closed.
        # fix: PREVIEW_MAX_* were local variables with width/height swapped
        while WEBCAM_RUNNING:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % 60 == 0:
                predict_frame(frame)
            frame_count += 1
            for frame_processor in frame_processors:
                frame = frame_processor.process_frame(source_face, frame)
            show_frame(frame, (960, 540))
            try:
                ROOT.update()
            except tkinter.TclError:
                break
    finally:
        WEBCAM_RUNNING = False
        cap.release()
        try:
            PREVIEW.withdraw()
        except tkinter.TclError:
            pass


In [ ]:
%%writefile /content/roop/roop/ui.json
{
  "CTk": {
    "fg_color": ["gray95", "gray10"]
  },
  "CTkToplevel": {
    "fg_color": ["gray95", "gray10"]
  },
  "CTkFrame": {
    "corner_radius": 6,
    "border_width": 0,
    "fg_color": ["gray90", "gray13"],
    "top_fg_color": ["gray85", "gray16"],
    "border_color": ["gray65", "gray28"]
  },
  "CTkButton": {
    "corner_radius": 6,
    "border_width": 0,
    "fg_color": ["#3a7ebf", "#1f538d"],
    "hover_color": ["#325882", "#14375e"],
    "border_color": ["#3E454A", "#949A9F"],
    "text_color": ["#DCE4EE", "#DCE4EE"],
    "text_color_disabled": ["gray74", "gray60"]
  },
  "CTkLabel": {
    "corner_radius": 0,
    "fg_color": "transparent",
    "text_color": ["gray14", "gray84"]
  },
  "CTkEntry": {
    "corner_radius": 6,
    "border_width": 2,
    "fg_color": ["#F9F9FA", "#343638"],
    "border_color": ["#979DA2", "#565B5E"],
    "text_color": ["gray14", "gray84"],
    "placeholder_text_color": ["gray52", "gray62"]
  },
  "CTkCheckbox": {
    "corner_radius": 6,
    "border_width": 3,
    "fg_color": ["#3a7ebf", "#1f538d"],
    "border_color": ["#3E454A", "#949A9F"],
    "hover_color": ["#325882", "#14375e"],
    "checkmark_color": ["#DCE4EE", "gray90"],
    "text_color": ["gray14", "gray84"],
    "text_color_disabled": ["gray60", "gray45"]
  },
  "CTkSwitch": {
    "corner_radius": 1000,
    "border_width": 3,
    "button_length": 0,
    "fg_color": ["#939BA2", "#4A4D50"],
    "progress_color": ["#3a7ebf", "#1f538d"],
    "button_color": ["gray36", "#D5D9DE"],
    "button_hover_color": ["gray20", "gray100"],
    "text_color": ["gray14", "gray84"],
    "text_color_disabled": ["gray60", "gray45"]
  },
  "CTkRadiobutton": {
    "corner_radius": 1000,
    "border_width_checked": 6,
    "border_width_unchecked": 3,
    "fg_color": ["#3a7ebf", "#1f538d"],
    "border_color": ["#3E454A", "#949A9F"],
    "hover_color": ["#325882", "#14375e"],
    "text_color": ["gray14", "gray84"],
    "text_color_disabled": ["gray60", "gray45"]
  },
  "CTkProgressBar": {
    "corner_radius": 1000,
    "border_width": 0,
    "fg_color": ["#939BA2", "#4A4D50"],
    "progress_color": ["#3a7ebf", "#1f538d"],
    "border_color": ["gray", "gray"]
  },
  "CTkSlider": {
    "corner_radius": 1000,
    "button_corner_radius": 1000,
    "border_width": 6,
    "button_length": 0,
    "fg_color": ["#939BA2", "#4A4D50"],
    "progress_color": ["gray40", "#AAB0B5"],
    "button_color": ["#3a7ebf", "#1f538d"],
    "button_hover_color": ["#325882", "#14375e"]
  },
  "CTkOptionMenu": {
    "corner_radius": 6,
    "fg_color": ["#3a7ebf", "#1f538d"],
    "button_color": ["#325882", "#14375e"],
    "button_hover_color": ["#234567", "#1e2c40"],
    "text_color": ["#DCE4EE", "#DCE4EE"],
    "text_color_disabled": ["gray74", "gray60"]
  },
  "CTkComboBox": {
    "corner_radius": 6,
    "border_width": 2,
    "fg_color": ["#F9F9FA", "#343638"],
    "border_color": ["#979DA2", "#565B5E"],
    "button_color": ["#979DA2", "#565B5E"],
    "button_hover_color": ["#6E7174", "#7A848D"],
    "text_color": ["gray14", "gray84"],
    "text_color_disabled": ["gray50", "gray45"]
  },
  "CTkScrollbar": {
    "corner_radius": 1000,
    "border_spacing": 4,
    "fg_color": "transparent",
    "button_color": ["gray55", "gray41"],
    "button_hover_color": ["gray40", "gray53"]
  },
  "CTkSegmentedButton": {
    "corner_radius": 6,
    "border_width": 2,
    "fg_color": ["#979DA2", "gray29"],
    "selected_color": ["#3a7ebf", "#1f538d"],
    "selected_hover_color": ["#325882", "#14375e"],
    "unselected_color": ["#979DA2", "gray29"],
    "unselected_hover_color": ["gray70", "gray41"],
    "text_color": ["#DCE4EE", "#DCE4EE"],
    "text_color_disabled": ["gray74", "gray60"]
  },
  "CTkTextbox": {
    "corner_radius": 6,
    "border_width": 0,
    "fg_color": ["gray100", "gray20"],
    "border_color": ["#979DA2", "#565B5E"],
    "text_color": ["gray14", "gray84"],
    "scrollbar_button_color": ["gray55", "gray41"],
    "scrollbar_button_hover_color": ["gray40", "gray53"]
  },
  "CTkScrollableFrame": {
    "label_fg_color": ["gray80", "gray21"]
  },
  "DropdownMenu": {
    "fg_color": ["gray90", "gray20"],
    "hover_color": ["gray75", "gray28"],
    "text_color": ["gray14", "gray84"]
  },
  "CTkFont": {
    "macOS": {
      "family": "Avenir",
      "size": 12,
      "weight": "normal"
    },
    "Windows": {
      "family": "Corbel",
      "size": 12,
      "weight": "normal"
    },
    "Linux": {
      "family": "Montserrat",
      "size": 12,
      "weight": "normal"
    }
  },
  "RoopDonate": {
    "text_color": ["#3a7ebf", "gray60"]
  }
}


In [ ]:
%%writefile /content/roop/roop/processors/frame/core.py
import os
import math
import importlib
import psutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from queue import Queue
from types import ModuleType
from typing import Any, List, Callable
from tqdm import tqdm

import roop.globals

FRAME_PROCESSORS_MODULES: List[ModuleType] = []
FRAME_PROCESSORS_INTERFACE = [
    'pre_check', 'pre_start', 'process_frame', 'process_frames',
    'process_image', 'process_video', 'post_process'
]


def load_frame_processor_module(frame_processor: str) -> Any:
    try:
        module = importlib.import_module(f'roop.processors.frame.{frame_processor}')
        for method_name in FRAME_PROCESSORS_INTERFACE:
            if not hasattr(module, method_name):
                raise NotImplementedError(method_name)
    except (ImportError, NotImplementedError) as error:
        quit(f'Frame processor {frame_processor} crashed: {error!r}')
    return module


def get_frame_processors_modules(frame_processors: List[str]) -> List[ModuleType]:
    global FRAME_PROCESSORS_MODULES

    if not FRAME_PROCESSORS_MODULES:
        for frame_processor in frame_processors:
            FRAME_PROCESSORS_MODULES.append(load_frame_processor_module(frame_processor))
    return FRAME_PROCESSORS_MODULES


def multi_process_frame(source_path: str, temp_frame_paths: List[str], process_frames: Callable[[str, List[str], Any], None], update: Callable[[], None]) -> None:
    threads = max(1, roop.globals.execution_threads or 1)
    # fix: floor division gave 0 when frames < threads -> infinite loop
    chunk_size = max(1, math.ceil(len(temp_frame_paths) / threads))
    with ThreadPoolExecutor(max_workers=threads) as executor:
        queue = create_queue(temp_frame_paths)
        futures = []
        while not queue.empty():
            futures.append(executor.submit(process_frames, source_path, pick_queue(queue, chunk_size), update))
        for future in as_completed(futures):
            future.result()


def create_queue(temp_frame_paths: List[str]) -> 'Queue[str]':
    queue: 'Queue[str]' = Queue()
    for frame_path in temp_frame_paths:
        queue.put(frame_path)
    return queue


def pick_queue(queue: 'Queue[str]', queue_per_future: int) -> List[str]:
    return [queue.get() for _ in range(queue_per_future) if not queue.empty()]


def process_video(source_path: str, frame_paths: List[str], process_frames: Callable[[str, List[str], Any], None]) -> None:
    progress_bar_format = '{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]'
    with tqdm(total=len(frame_paths), desc='Processing', unit='frame', dynamic_ncols=True, bar_format=progress_bar_format) as progress:
        multi_process_frame(source_path, frame_paths, process_frames, lambda: update_progress(progress))


def update_progress(progress: Any = None) -> None:
    memory_usage = psutil.Process(os.getpid()).memory_info().rss / 1024 ** 3
    progress.set_postfix({
        'memory_usage': f'{memory_usage:05.2f}GB',
        'execution_providers': roop.globals.execution_providers,
        'execution_threads': roop.globals.execution_threads
    })
    progress.update(1)


In [ ]:
%%writefile /content/roop/roop/processors/frame/face_swapper.py
from typing import Any, List, Callable
import cv2
import insightface
import threading

import roop.globals
import roop.processors.frame.core
from roop.core import update_status
from roop.face_analyser import get_one_face, get_many_faces
from roop.typing import Face, Frame
from roop.utilities import conditional_download, resolve_relative_path, is_image, is_video

FACE_SWAPPER = None
THREAD_LOCK = threading.Lock()
NAME = 'ROOP.FACE-SWAPPER'


def get_face_swapper() -> Any:
    global FACE_SWAPPER

    with THREAD_LOCK:
        if FACE_SWAPPER is None:
            model_path = resolve_relative_path('../models/inswapper_128.onnx')
            FACE_SWAPPER = insightface.model_zoo.get_model(model_path, providers=roop.globals.execution_providers)
    return FACE_SWAPPER


def pre_check() -> bool:
    download_directory_path = resolve_relative_path('../models')
    conditional_download(download_directory_path, ['https://huggingface.co/countfloyd/deepfake/resolve/main/inswapper_128.onnx'])
    return True


def pre_start() -> bool:
    if not is_image(roop.globals.source_path):
        update_status('Select an image for source path.', NAME)
        return False
    source = cv2.imread(roop.globals.source_path)
    if source is None or not get_one_face(source):
        update_status('No face in source path detected.', NAME)
        return False
    if not is_image(roop.globals.target_path) and not is_video(roop.globals.target_path):
        update_status('Select an image or video for target path.', NAME)
        return False
    return True


def post_process() -> None:
    global FACE_SWAPPER

    FACE_SWAPPER = None


def swap_face(source_face: Face, target_face: Face, temp_frame: Frame) -> Frame:
    return get_face_swapper().get(temp_frame, target_face, source_face, paste_back=True)


def process_frame(source_face: Face, temp_frame: Frame) -> Frame:
    if source_face is None:
        return temp_frame
    if roop.globals.many_faces:
        if many_faces := get_many_faces(temp_frame):
            for target_face in many_faces:
                temp_frame = swap_face(source_face, target_face, temp_frame)
    elif target_face := get_one_face(temp_frame):
        temp_frame = swap_face(source_face, target_face, temp_frame)
    return temp_frame


def process_frames(source_path: str, temp_frame_paths: List[str], update: Callable[[], None]) -> None:
    source_face = get_one_face(cv2.imread(source_path))
    for temp_frame_path in temp_frame_paths:
        temp_frame = cv2.imread(temp_frame_path)
        if temp_frame is not None:  # fix: unreadable frame used to crash the worker
            cv2.imwrite(temp_frame_path, process_frame(source_face, temp_frame))
        if update:
            update()


def process_image(source_path: str, target_path: str, output_path: str) -> None:
    source_face = get_one_face(cv2.imread(source_path))
    target_frame = cv2.imread(target_path)
    cv2.imwrite(output_path, process_frame(source_face, target_frame))


def process_video(source_path: str, temp_frame_paths: List[str]) -> None:
    roop.processors.frame.core.process_video(source_path, temp_frame_paths, process_frames)


In [ ]:
%%writefile /content/roop/roop/processors/frame/face_enhancer.py
from typing import Any, List, Callable
import cv2
import threading
import gfpgan

import roop.globals
import roop.processors.frame.core
from roop.core import update_status
from roop.typing import Frame, Face
from roop.utilities import conditional_download, resolve_relative_path, is_image, is_video

FACE_ENHANCER = None
THREAD_SEMAPHORE = threading.Semaphore()
THREAD_LOCK = threading.Lock()
NAME = 'ROOP.FACE-ENHANCER'


def get_face_enhancer() -> Any:
    global FACE_ENHANCER

    with THREAD_LOCK:
        if FACE_ENHANCER is None:
            model_path = resolve_relative_path('../models/GFPGANv1.4.pth')
            FACE_ENHANCER = gfpgan.GFPGANer(model_path=model_path, upscale=1)  # type: ignore[attr-defined]
    return FACE_ENHANCER


def pre_check() -> bool:
    download_directory_path = resolve_relative_path('../models')
    conditional_download(download_directory_path, ['https://huggingface.co/henryruhs/roop/resolve/main/GFPGANv1.4.pth'])
    return True


def pre_start() -> bool:
    if not is_image(roop.globals.target_path) and not is_video(roop.globals.target_path):
        update_status('Select an image or video for target path.', NAME)
        return False
    return True


def post_process() -> None:
    global FACE_ENHANCER

    FACE_ENHANCER = None


def enhance_face(temp_frame: Frame) -> Frame:
    with THREAD_SEMAPHORE:
        _, _, result = get_face_enhancer().enhance(temp_frame, paste_back=True)
    return temp_frame if result is None else result


def process_frame(source_face: Face, temp_frame: Frame) -> Frame:
    # efficiency: GFPGAN detects faces itself; the extra insightface
    # detection pass per frame was redundant and doubled the cost
    return enhance_face(temp_frame)


def process_frames(source_path: str, temp_frame_paths: List[str], update: Callable[[], None]) -> None:
    for temp_frame_path in temp_frame_paths:
        temp_frame = cv2.imread(temp_frame_path)
        if temp_frame is not None:
            cv2.imwrite(temp_frame_path, process_frame(None, temp_frame))
        if update:
            update()


def process_image(source_path: str, target_path: str, output_path: str) -> None:
    target_frame = cv2.imread(target_path)
    cv2.imwrite(output_path, process_frame(None, target_frame))


def process_video(source_path: str, temp_frame_paths: List[str]) -> None:
    roop.processors.frame.core.process_video(None, temp_frame_paths, process_frames)


## 3. Run

In [ ]:
#@title Options
MANY_FACES = False #@param {type:"boolean"}
ENHANCE = False #@param {type:"boolean"}  # needs the optional GFPGAN cell
KEEP_AUDIO = True #@param {type:"boolean"}
VIDEO_QUALITY = 18 #@param {type:"slider", min:0, max:51, step:1}

OUTPUT = '/content/output.mp4'
procs = 'face_swapper face_enhancer' if ENHANCE else 'face_swapper'
flags = ('--many-faces ' if MANY_FACES else '') + ('--keep-audio' if KEEP_AUDIO else '--no-keep-audio')
!cd /content/roop && python run.py -s "/content/face1.jpg" -t "/content/video1.mp4" -o "/content/output.mp4" --frame-processor {procs} {flags} --video-quality {VIDEO_QUALITY}
print("\nDone! Your result is /content/output.mp4 - open the Files panel (left), click refresh, then right-click the file > Download.")